# 02 · An MLP from scratch: solving XOR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/nn-fundamentals/02_mlp_numpy.ipynb) &nbsp; 📖 Concept deck: [From perceptron to MLP](https://nilesh-salpe.github.io/visual-ai/nn-fundamentals/01-perceptron-to-mlp/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** why a linear model cannot fit XOR, a two-layer MLP with a hidden nonlinearity, its hand-written backward pass, and training it to 100% accuracy.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
Y = np.array([0, 1, 1, 0], float)        # XOR
def sigmoid(z): return 1 / (1 + np.exp(-z))

## Task 1: A linear model cannot do XOR

Try many random linear classifiers $\hat y = 1[w \cdot x + b > 0]$ (say 20,000 draws of w and b from a standard normal) and return the **best accuracy** any of them gets on XOR.
No line separates {(0,1), (1,0)} from {(0,0), (1,1)}, so the best is 3/4.

In [ ]:
def best_linear_accuracy(n=20000, seed=0):
    r = np.random.default_rng(seed)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_best_linear_accuracy(f):
    acc = f()
    assert np.isclose(acc, 0.75), f"best linear accuracy on XOR should be 0.75, got {acc}"

check("best_linear_accuracy", _test_best_linear_accuracy, best_linear_accuracy)

## Task 2: Forward and backward of a 2-layer MLP

Model: $h = \tanh(X W_1 + b_1)$, $p = \sigma(h W_2 + b_2)$, binary cross-entropy $L = -\frac{1}{n}\sum [y\log p + (1-y)\log(1-p)]$.
Return `(L, grads)` with `grads = dict(W1=..., b1=..., W2=..., b2=...)`. Useful facts: $\partial L/\partial z_2 = (p - y)/n$ for the output pre-activation; $\tanh' = 1 - h^2$.

In [ ]:
def mlp_forward_backward(params, X, Y):
    W1, b1, W2, b2 = params['W1'], params['b1'], params['W2'], params['b2']
    # TODO: forward, loss, backward
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_mlp_forward_backward(f):
    r = np.random.default_rng(1)
    P = dict(W1=r.normal(size=(2, 4)), b1=r.normal(size=4), W2=r.normal(size=(4, 1)), b2=r.normal(size=1))
    L, g = f(P, X, Y)
    assert set(g) == {'W1', 'b1', 'W2', 'b2'} and all(g[k].shape == P[k].shape for k in P)
    for k in P:
        Pp = {q: v.copy() for q, v in P.items()}; Pp[k].flat[0] += 1e-6
        num = (f(Pp, X, Y)[0] - L) / 1e-6
        assert abs(num - g[k].flat[0]) < 1e-4, f"grad of {k} disagrees with a finite difference ({g[k].flat[0]:.5f} vs {num:.5f})"

check("mlp_forward_backward", _test_mlp_forward_backward, mlp_forward_backward)

## Task 3: Train it

Initialise W₁ (2 × hidden), W₂ (hidden × 1) from a normal with std 1, biases at 0, and run full-batch gradient descent with learning rate `lr` for `steps` steps.
Return `(losses, accuracy)`.

In [ ]:
def train_xor(hidden=8, lr=1.0, steps=2000, seed=0):
    r = np.random.default_rng(seed)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_train_xor(f):
    losses, acc = f()
    assert acc == 1.0, f"the MLP should solve XOR, accuracy {acc}"
    assert losses[-1] < 0.05 and losses[-1] < losses[0]

check("train_xor", _test_train_xor, train_xor)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: A linear model cannot do XOR
def best_linear_accuracy_solution(n=20000, seed=0):
    r = np.random.default_rng(seed)
    W, B = r.normal(size=(n, 2)), r.normal(size=n)
    pred = (X @ W.T + B) > 0
    return ((pred == Y[:, None]).mean(axis=0)).max()

check("best_linear_accuracy (reference)", _test_best_linear_accuracy, best_linear_accuracy_solution)

In [ ]:
# @title Solution for Task 2: Forward and backward of a 2-layer MLP
def mlp_forward_backward_solution(params, X, Y):
    W1, b1, W2, b2 = params['W1'], params['b1'], params['W2'], params['b2']
    n = len(X)
    h = np.tanh(X @ W1 + b1); p = sigmoid(h @ W2 + b2)[:, 0]
    L = -np.mean(Y * np.log(p) + (1 - Y) * np.log(1 - p))
    dz2 = ((p - Y) / n)[:, None]
    dW2, db2 = h.T @ dz2, dz2.sum(0)
    dz1 = (dz2 @ W2.T) * (1 - h ** 2)
    return L, dict(W1=X.T @ dz1, b1=dz1.sum(0), W2=dW2, b2=db2)

check("mlp_forward_backward (reference)", _test_mlp_forward_backward, mlp_forward_backward_solution)

In [ ]:
# @title Solution for Task 3: Train it
def train_xor_solution(hidden=8, lr=1.0, steps=2000, seed=0):
    r = np.random.default_rng(seed)
    P = dict(W1=r.normal(size=(2, hidden)), b1=np.zeros(hidden), W2=r.normal(size=(hidden, 1)), b2=np.zeros(1))
    losses = []
    for _ in range(steps):
        L, g = mlp_forward_backward_solution(P, X, Y); losses.append(L)
        for k in P: P[k] -= lr * g[k]
    h = np.tanh(X @ P['W1'] + P['b1']); p = sigmoid(h @ P['W2'] + P['b2'])[:, 0]
    return losses, ((p > 0.5) == Y).mean()

check("train_xor (reference)", _test_train_xor, train_xor_solution)

---
## Experiments
**How many hidden units?** Two are enough in principle for XOR, but optimisation from a random start succeeds more reliably with a few more. Count successes over seeds.

In [ ]:
for hdim in [1, 2, 3, 4, 8]:
    ok = sum(train_xor_solution(hidden=hdim, seed=s)[1] == 1.0 for s in range(20))
    print(f"hidden={hdim}: solved XOR in {ok}/20 random starts")

---
## Takeaways
* A single linear layer draws one straight boundary; XOR needs two.
* A hidden layer with a nonlinearity bends the input space so the output layer can separate it.
* The backward pass is a few matmuls and elementwise products, exactly the chain rule.